# Calibration for fine-tuned prognostic risk prediction

In [1]:
experiment = "cvd" 
pre_trained_model = "SurvivEHR-cr-small-debug7_exp1000-v1-v4-v1"
seed = 1

In [2]:
import os
from pathlib import Path
import sys
node_type = os.getenv('BB_CPU')
venv_dir = f'/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-{node_type}'
venv_site_pkgs = Path(venv_dir) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
if venv_site_pkgs.exists():
    sys.path.insert(0, str(venv_site_pkgs))
    print(f"Added path '{venv_site_pkgs}' at start of search paths.")
else:
    print(f"Path '{venv_site_pkgs}' not found. Check that it exists and/or that it exists for node-type '{node_type}'.")

%load_ext autoreload
%autoreload 2

print(os.getcwd())

Added path '/rds/homes/g/gaddcz/Projects/CPRD/virtual-envTorch2.0-icelake/lib/python3.10/site-packages' at start of search paths.
/rds/homes/g/gaddcz/Projects/CPRD/examples/modelling/SurvivEHR/notebooks/CompetingRisk/4_Calibration


In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import logging
import wandb
import pickle
from hydra import compose, initialize
from omegaconf import OmegaConf

from FastEHR.dataloader import FoundationalDataModule

from SurvivEHR.examples.modelling.SurvivEHR.run_experiment import run
from SurvivEHR.src.models.survival.task_heads.causal import SurvStreamGPTForCausalModelling

import time
import polars as pl
pl.Config.set_tbl_rows(10000)
import pandas as pd
pd.options.display.max_rows = 10000


# TODO: define an env variable to fix for a local hpc environment issue, this shouldn't be needed
%env SLURM_NTASKS_PER_NODE=28   

env: SLURM_NTASKS_PER_NODE=28


In [4]:
match experiment.lower():
    case "cvd":
        experiment_type = "fine-tune-cr"
        fine_tune_model_name = f"cvd-fine-tune-cr-Afalse8-Nsnull-s{seed}"
    case "hypertension":
        experiment_type = "fine-tune-sr"
        # fine_tune_model_name = f"NorthEast{seed}"

In [5]:
 # load the configuration file, override any settings 
with initialize(version_base=None, config_path="../../../confs", job_name="testing_notebook"):
    cfg = compose(config_name="config_CompetingRisk11M", 
                  overrides=[# Experiment setup
                             f"experiment.type='{experiment_type}'",
                             f"experiment.run_id='{pre_trained_model}'",
                             f"experiment.fine_tune_id='{fine_tune_model_name}'",
                             "experiment.train=False",
                             "experiment.test=False",
                             f"experiment.seed={seed}",
                             # Dataloader
                             "data.batch_size=128",
                             "data.meta_information_path=/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/PreTrain/meta_information_QuantJenny.pickle",
                             "data.min_workers=12",
                             "data.global_diagnoses=True",
                             "data.repeating_events=False",
                             # Model
                             "transformer.block_size=512", 
                            ]
                 )

match experiment.lower():
    case "cvd":
        cfg.data.path_to_ds="/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/FineTune_CVD/"
        cfg.fine_tuning.fine_tune_outcomes=["IHDINCLUDINGMI_OPTIMALV2", "ISCHAEMICSTROKE_V2", "MINFARCTION", "STROKEUNSPECIFIED_V2", "STROKE_HAEMRGIC"]
        
    case "hypertension":
        cfg.data.path_to_ds="/rds/projects/g/gokhalkm-optimal/OPTIMAL_MASTER_DATASET/data/FoundationalModel/ByRegion/Hypertension_North East/"
        cfg.fine_tuning.fine_tune_outcomes=["HYPERTENSION"]
    
experiment, dm = run(cfg)
wandb.finish()

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [18]:
def stratify_batch(batch, age_intervals=None):

    # By default set age intervals in 5 years, from 40-45, to 75-80.
    if age_intervals is None:
        age_intervals = [break_point for break_point in range(40, 85, 5)]

    static_covariates = dm.test_set._decode_covariates(batch["static_covariates"])
    
    display(static_covariates["SEX"])
    display(static_covariates["birth_year"])
    

In [19]:
for batch in dm.test_dataloader():
    break
print(stratify_batch(batch))
    

array(['M', 'F', 'F', 'F', 'F', 'M', 'F', 'F', 'F', 'M', 'M', 'F', 'F',
       'M', 'M', 'F', 'F', 'M', 'F', 'F', 'F', 'F', 'M', 'M', 'F', 'F',
       'F', 'M', 'M', 'F', 'M', 'F', 'F', 'F', 'M', 'F', 'M', 'F', 'M',
       'F', 'F', 'M', 'M', 'F', 'F', 'M', 'F', 'F', 'F', 'M', 'M', 'M',
       'M', 'M', 'F', 'F', 'F', 'M', 'M', 'F', 'M', 'M', 'F', 'M', 'M',
       'F', 'F', 'M', 'F', 'F', 'M', 'M', 'F', 'F', 'M', 'M', 'F', 'F',
       'M', 'F', 'M', 'M', 'M', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'M',
       'F', 'M', 'F', 'M', 'M', 'F', 'M', 'F', 'M', 'F', 'M', 'M', 'F',
       'F', 'M', 'M', 'F', 'M', 'F', 'F', 'F', 'M', 'F', 'F', 'M', 'M',
       'M', 'F', 'F', 'M', 'F', 'F', 'F', 'M', 'F', 'M', 'M'],
      dtype=object)

tensor([1968., 1965., 1922., 1969., 1942., 1967., 1933., 1952., 1935., 1938.,
        1949., 1962., 1972., 1936., 1982., 1961., 1950., 1955., 1941., 1967.,
        1962., 1974., 1950., 1950., 1936., 1957., 1960., 1965., 1948., 1969.,
        1944., 1947., 1944., 1927., 1932., 1945., 1970., 1974., 1954., 1978.,
        1938., 1967., 1967., 1932., 1961., 1956., 1931., 1959., 1959., 1939.,
        1967., 1939., 1959., 1963., 1933., 1940., 1969., 1972., 1964., 1932.,
        1978., 1945., 1942., 1956., 1970., 1980., 1964., 1954., 1953., 1973.,
        1966., 1962., 1954., 1942., 1975., 1936., 1953., 1945., 1925., 1935.,
        1924., 1974., 1945., 1934., 1921., 1976., 1960., 1935., 1956., 1944.,
        1959., 1954., 1957., 1938., 1955., 1983., 1932., 1964., 1959., 1969.,
        1939., 1931., 1962., 1928., 1963., 1933., 1961., 1946., 1968., 1975.,
        1943., 1932., 1958., 1948., 1963., 1961., 1962., 1958., 1971., 1928.,
        1943., 1941., 1985., 1964., 1942., 1947., 1985., 1963.])

None


In [25]:
batch["ages"]

tensor([[ 5.2630,  7.1107,  7.6312,  ...,  0.0000,  0.0000,  0.0000],
        [ 5.2964,  5.6970,  6.3123,  ...,  0.0000,  0.0000,  0.0000],
        [12.9019, 14.1030, 14.5030,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [10.0455, 10.0455, 11.7655,  ...,  0.0000,  0.0000,  0.0000],
        [ 1.6942,  4.0959,  4.0959,  ...,  0.0000,  0.0000,  0.0000],
        [10.5863, 10.5995, 10.5995,  ...,  0.0000,  0.0000,  0.0000]])